# Databricks Lakeflow Jobs — Complete Feature Guide

This notebook is a detailed, presentation-ready walkthrough of **Lakeflow Jobs**: triggers & scheduling, compute & environments, task orchestration, parameters & dynamic values, reliability & repair, monitoring & alerting, and governance & CI/CD. Each feature is explained with **Configuration** (where you set it up) and **Runtime behavior** (what exactly happens when the job runs).

## 1. Triggers & Scheduling

A job runs **manually** by default. In the job UI: **Tasks tab → Add trigger → New trigger** dialog (Status: Active, Type: dropdown). The Type dropdown offers: **Scheduled, File arrival, Table update, Continuous, Model update, Upstream job completion** — each maps to a `trigger` block in the Jobs API / `resources.jobs.<name>` in a Declarative Automation Bundle.

### 1.1 Scheduled (cron)
- **Configuration:** New trigger → Type *Scheduled* → **Schedule type** (specific time of day, or periodic interval), timezone, cron expression (e.g. `0 6 * * MON-FRI` = 06:00 every weekday). *Advanced configuration* adds **pause/quiesce windows** (e.g. freeze runs 02:00–03:00 for maintenance).
- **Runtime behavior:** At each cron tick Databricks starts a run. If a scheduled run is still in flight when the next tick fires, the new run is **skipped** unless **Maximum concurrent runs** > 1 (see 1.6). Changing the schedule affects only *future* triggers — an in-flight run is unaffected.

### 1.2 File arrival
- **Configuration:** New trigger → Type *File arrival* → point at a Unity Catalog Volume or cloud storage location, optionally a **glob pattern** (e.g. `incoming/*.csv`), plus a **minimum time between triggers** to absorb bursts of files.
- **Runtime behavior:** Databricks watches the location; the moment a **new file** arrives, a run starts (within seconds). Files landing *during* an active run either queue a new run (subject to max concurrent runs) or are ignored until the run completes, depending on the retrigger setting. This replaces "poll every hour for new CSVs" — ingestion starts when data actually lands.

### 1.3 Table update
- **Configuration:** New trigger → Type *Table update* → select one or more source tables (Delta/UC), optionally with a minimum time between triggered runs.
- **Runtime behavior:** When the source table's commit log advances (an append/update commits), the job is triggered. Typical pattern: a Spark Declarative Pipeline writes `bronze.orders` → this job (silver/gold build) fires the instant bronze updates — no guessed cron delay.

### 1.4 Continuous
- **Configuration:** New trigger → Type *Continuous* — "Start a new run as soon as the previous run finishes." Designed for streaming / near-real-time workloads (Databricks recommends it for production Structured Streaming).
- **Runtime behavior:** Exactly **one run active at a time**; when a run ends, a new one starts **immediately** — zero gap. Failures use **exponential backoff**: a failed task retries with increasing delay; after max retries (3 for a single-task job) the run is canceled and a fresh run is triggered. If consecutive failures keep stacking, the retry period keeps increasing up to a system maximum, and the job keeps retrying at that cadence **indefinitely** — the moment a run succeeds, backoff resets to healthy. You can manually restart a backoff-stuck job from the UI or a run-now call.

### 1.5 Model update & Upstream job completion
- **Model update:** Type *Model update* → select monitored UC registered models. Runs when a monitored model is **created or updated** — e.g. trigger a batch-scoring job the moment a newly trained model version lands in the registry.
- **Upstream job completion:** Type *Upstream job completion* → reference another job in the workspace, optionally filtering on its result state (success/failure). Runs the moment the upstream run reaches a terminal state and matches the filter — chaining jobs **across teams** without one giant monolithic job.

### 1.6 Maximum concurrent runs
- **Configuration:** Job **Settings → Advanced → Edit maximum concurrent runs** (default **1**).
- **Runtime behavior:** Guards against overlapping runs (e.g. an overrunning 2 AM run colliding with the 3 AM tick). With `1`, a new trigger while a run is active **skips** the run (and sends a 'skipped' notification unless muted). Raise it (e.g. 3) to allow parallel runs — each run gets its **own** job clusters, parameters, and task-value context; runs never share state.

### Trigger summary

| Trigger | Fires when | Typical use |
| --- | --- | --- |
| Scheduled | Fixed cron / interval | Nightly ETL, weekly reports |
| File arrival | New file in a volume/location | Landing-zone ingestion |
| Table update | Source table commits | Pipeline → downstream build |
| Continuous | Previous run finishes | Streaming, low-latency processing |
| Model update | Monitored model changes | Batch scoring on new model version |
| Upstream job completion | Another job's run ends | Cross-team dependency chains |

## 2. Compute & Environments

In each task's **Compute** settings you choose where the task runs: **Default Automated Compute (Serverless)**, a classic **Job cluster**, or an **Existing all-purpose cluster** (interactive/debug only). The Compute panel's *Configure* / *Swap* buttons switch between them.

### 2.1 Serverless compute (Default Automated Compute)
- **Configuration:** Leave the task's compute on *Default Automated Compute (Serverless)*, or select a specific serverless compute object / policy under **Swap**. Serverless jobs **auto-optimize retries by default**.
- **Runtime behavior:** No cluster cold-start — tasks run on pre-warmed, Databricks-managed compute, so runs start in seconds and you pay only while tasks execute. Job clusters, by contrast, spin up at run start (minutes of startup cost) and terminate after the run.

### 2.2 Environment specifications (dependencies)
- **Configuration:** Job → **Environments** → *Configure environment*: Name (e.g. `Default`), Environment source (*Standard* or *Project / pyproject.toml*), Base environment (e.g. **Standard v2** — Python 3.11, Scala 2.12), and **Dependencies** (pip-style entries like `simplejson==3.19.*` or `-r requirements.txt`). Tasks reference the environment by key; on serverless it is **required** for Python script / wheel / dbt tasks. Alternatively, classic clusters take **Dependent libraries** per task (Cluster libraries mode).
- **Runtime behavior:** Databricks resolves the environment with exactly those dependencies before the task runs — no `%pip install` cells, no per-run driver install time, identical library versions on every run. If a pinned dependency can't be resolved, the task fails fast at environment setup rather than mid-notebook.

## 3. Task Orchestration & Control Flow

### 3.1 Task dependencies (Depends on / Run if)
- **Configuration:** Each task's **Depends on** field in the Tasks graph; **Run if** picks the condition: *All succeeded* (default), *At least one succeeded*, *None failed*, *All done*, *At least one failed*, *All failed*.
- **Runtime behavior:** Databricks runs tasks topologically, **in parallel wherever the graph allows**. `Run if` gates on upstream outcomes: a cleanup task with *All done* runs whether ingest succeeded or failed; an error-handler with *At least one failed* runs only on failure. When an upstream task fails, downstream tasks are **skipped** (shown grayed in the run view), not failed.

### 3.2 If/else condition task
- **Configuration:** Add an **If/else** task; write a boolean expression over dynamic values, e.g. `{{tasks.validate.values.row_count}} > 0`. Downstream tasks attach to either `<task> (true)` or `<task> (false)`.
- **Runtime behavior:** The expression evaluates once at run time; only the matching branch's tasks execute — the other branch is **skipped**. Classic scenario: if the ingest found no new rows, skip the expensive transform/retrain branch and run the notify/exit branch instead.

### 3.3 For each loop
- **Configuration:** Add a **For each** task; give it an input array (static list, or dynamic like `{{tasks.scan.outputs.tables_to_process}}`), a nested task (notebook / SQL / Run Job) whose parameters use `{{input}}`-style references, and a **concurrency** (iterations in parallel).
- **Runtime behavior:** One instance of the nested task per array element, up to the concurrency limit. E.g. 20 tables with concurrency 5 → 5 parallel instances, a new one starting as each finishes. If one iteration fails, the For each task is marked failed (subject to its own retries) while other iterations run to completion.

### 3.4 Run Job task (modular sub-workflows)
- **Configuration:** Add a **Run Job** task pointing at another job in the workspace, optionally passing parameters.
- **Runtime behavior:** The parent run pauses on that task until the child job's run reaches a terminal state; the child's result surfaces as the task result. This creates reusable "library" jobs (e.g. a shared *validate & register dataset* job) invoked from many parents, each with its own run history, schedule, and permissions.

## 4. Parameters & Dynamic Values

### 4.1 Job parameters
- **Configuration:** Job → **Parameters** section: key/value pairs with optional defaults.
- **Runtime behavior:** Defaults live on the job, but every trigger path can override them: **Run now with different values**, REST API trigger, or parent-job → child-job parameters. Tasks consume them via `{{job.parameters.<key>}}`. One job can therefore process `env=dev` in testing and `env=prod` at the 2 AM cron — no cloning.

### 4.2 Task values (dbutils.jobs.taskValues)
- **Configuration:** Inside a task: `dbutils.jobs.taskValues.set(key="row_count", value=84213)`. Values must be small scalars/JSON-serializable.
- **Runtime behavior:** The value is written to the run context at task completion; any **later** task in the same run reads it via `dbutils.jobs.taskValues.get(taskKey="ingest", key="row_count")` or the dynamic reference `{{tasks.ingest.values.row_count}}`. No shared table or temp file needed. (In a repair run, references resolve from the original run's context.)

### 4.3 Dynamic value references
Placeholders resolve **at run time, just before the consuming task starts**, to actual run metadata. In the UI, fields that accept them show the `{ }` insert shortcut.

| Reference | Resolves to |
| --- | --- |
| `{{job.id}}` / `{{job.name}}` | The job's id / name |
| `{{job.run_id}}` / `{{task.run_id}}` | Run / task attempt id |
| `{{job.start_time.iso_date}}` | Run start date (e.g. for `daily_{{...}}` paths) |
| `{{job.parameters.table}}` | A job parameter |
| `{{tasks.ingest.values.row_count}}` | A task value set by an earlier task |
| `{{tasks.transform.output.table_name}}` | Structured output of some task types |
| Retry/repair references | Task retry count, repair count |

They are substituted as plain text before execution — e.g. a dbt task command `dbt run --vars '{"date": "{{job.start_time.iso_date}}"}'`. Note: a reference to a task value only resolves if that upstream task actually **ran in this run**.

The cell below demonstrates setting/reading a task value safely (it degrades gracefully when the notebook isn't running inside a job task).

In [0]:
# Demo: passing data between tasks with task values + dynamic values.
# When this notebook runs as a JOB TASK, dbutils.jobs.taskValues writes into the
# run context and later tasks read it; dynamic value references like
# {{job.id}} / {{tasks.<task>.values.<key>}} are resolved by the Jobs service
# before the task starts.

# 1) Publish a task value (works only inside a job run):
try:
    dbutils.jobs.taskValues.set(key="row_count", value=84213)
    dbutils.jobs.taskValues.set(key="data_available", value=True)
    print("task values published: row_count=84213, data_available=True")
except Exception as e:
    print("Not running inside a job task -> taskValues.set unavailable:", type(e).__name__)

# 2) Read a task value published by an upstream task in the same run:
try:
    rc = dbutils.jobs.taskValues.get(taskKey="ingest", key="row_count", debugValue=0)
    print("upstream 'ingest' row_count =", rc)
except Exception as e:
    print("No upstream task value in interactive mode ->", type(e).__name__)

# 3) In an If/else condition task you would write the boolean expression as:
#    {{tasks.ingest.values.row_count}} > 0
# 4) In a downstream task parameter you would write:
#    processed_{{job.start_time.iso_date}}  ->  e.g. processed_2026-11-27

## 5. Reliability, Retries & Repair

### 5.1 Task retry policies
- **Configuration:** Task pane → **Retries → Add**: retry count, plus either a fixed **interval** or exponential backoff with min/max interval. Default: **no retries** for most classic configs; **serverless jobs auto-optimize retries by default**; continuous jobs use exponential backoff inherently.
- **Runtime behavior:** On task failure the engine schedules the retry (interval measured from the failed run's start); the task re-runs up to N times. Only after the final retry fails is the task marked failed and downstream tasks skipped. If both **Timeout** and **Retries** are set, the timeout applies **to each retry attempt individually**.

### 5.2 Duration thresholds: warning vs. timeout
- **Configuration:** Task (or job) settings → **Duration warning** (expected completion time) and **Timeout** (maximum completion time). The warning must be < the timeout.
- **Runtime behavior — the crucial distinction:**
  * **Exceeds the warning threshold** → the task is **NOT stopped**. A *Duration warning* notification is sent to configured destinations (if they subscribe to that event) and the task keeps running.
  * **Exceeds the timeout** → the task is **killed**: run state becomes `TIMED_OUT`, resources are released, downstream tasks are skipped (per their Run-if conditions).
  * Scenario: ingest normally takes 10 min → warning 15 min, timeout 60 min. A hung SFTP read triggers a Slack "slow job" ping at 15 min while still attempting; at 60 min the task is terminated and on-call gets a failure notification.

### 5.3 Repair and rerun failed tasks
- **Configuration:** On a failed run's details page → **Repair run** (or the Jobs API repair endpoint). The dialog lists failed + skipped tasks; you can override parameters and edit job/task settings first.
- **Runtime behavior:** **Only the unsuccessful tasks and their dependents re-run — succeeded tasks are NOT re-executed.** Repaired tasks run against the *current* settings (fix the notebook, then repair). Critical caveats:
  * Each unsuccessful task restarts **from the beginning** — Jobs does not make tasks idempotent, so a task that half-appended data before failing can duplicate it. Prefer overwrite/MERGE semantics.
  * Shared job clusters are **re-created as new versioned clusters** (e.g. `my_cluster_v1`) on repair.
  * Only **multi-task** jobs support repair; a failed single-task job is simply "Run now" again.
  * The run's displayed **Duration spans original run → last repair completion**.
  * Repair uses each task's **run state**, not its disabled state — a disabled-but-failed task is still a repair candidate unless explicitly excluded.

## 6. Monitoring, Thresholds & Alerting

### 6.1 Notifications (email / Slack / PagerDuty / Teams / webhooks)
- **Configuration:** Job → **Job notifications → Edit** → add destinations (raw email addresses, or admin-defined **system destinations**: Slack, PagerDuty, Microsoft Teams, generic webhooks) and check event types: **Start, Success, Failure, Duration warning, Streaming backlog**. Max **three system destinations per event type**. Task-level notifications are configured separately in the task pane.
- **Runtime behavior:**
  * **Job-level** notifications fire on the run's terminal state — and notably **don't fire when failed tasks are retried**; use *task-level* notifications for a ping per failed attempt.
  * **"Succeeded with failures"** counts as success — check *Success* to hear about it.
  * **Muting:** mute skipped runs, canceled runs, and (task-level) until the last retry — cutting cascading-failure noise.
  * **Webhook destinations** receive a JSON payload (run ID, state, job name) for incident tooling; for stable schemas prefer user-defined webhooks over parsing Slack/Teams message text.

### 6.2 Job health configuration (metric thresholds + data quality)
- **Configuration:** Job → **Job health configuration**: **Metric thresholds** — *Duration warning* (e.g. 15m 0s) plus streaming **backlog thresholds** (files/bytes) — and **Data quality monitoring** (*Enable monitoring*), which tracks freshness and completeness of the job's output tables. Optionally add free-text **health rules** describing expected behavior (e.g. "alert if no successful run in 24h" or "failure rate > 20% over last 10 runs").
- **Runtime behavior:** Metric thresholds raise *Duration warning* / *Streaming backlog* notifications when breached — without killing the run (that's the separate timeout). Health rules are evaluated continuously against run history; a violation surfaces an **unhealthy** status in the Jobs UI — catching silent problems (like a cron job that stopped triggering) that per-run failure alerts never see. Data quality monitoring reports freshness/completeness signals for output tables.

### 6.3 System tables for monitoring
- **Configuration/usage:** Query the `system.lakeflow` tables — `job_run_timeline` (every run state change with timestamps, run-as identity) and `jobs` (current job metadata) — joinable to `system.billing.usage` via `usage_metadata.job_id`/`job_run_id` for **cost of failed runs** and **repair-heavy jobs** (a run_id with multiple state rows = repeated repairs).
- **Runtime behavior:** The cells below query these tables live — recent runs, failure ratios over 30 days, and per-job failure cost.

In [0]:
%sql
-- Live demo: your workspace's most recent job runs, newest first.
SELECT job_id,
       run_id,
       result_state,
       termination_code,
       period_start_time,
       period_end_time
FROM system.lakeflow.job_run_timeline
WHERE result_state IS NOT NULL
ORDER BY period_start_time DESC
LIMIT 10

## 7. Governance & CI/CD

### 7.1 Run as Service Principal
- **Configuration:** Job **Settings → Run as** → pick a service principal. Whoever sets it needs the **Service Principal User** role on that SP (Manager does NOT imply User) plus CAN MANAGE/owner on the job.
- **Runtime behavior:** All tasks execute with the SP's identity — its UC grants, its cluster-policy access. Production runs become independent of any human: if the creator leaves or loses table access, the job **keeps working**. If the SP lacks `SELECT` on a table, the run fails as the SP — the editor's own permissions are irrelevant at run time.

### 7.2 Unity Catalog permissions
- **Configuration:** Grants on catalogs/schemas/tables/volumes/warehouses for the run-as identity; job-level permissions (CAN VIEW / CAN RUN / CAN MANAGE) for the team.
- **Runtime behavior:** Every credential/permission check during the run is against the run-as principal. Production pattern: the team has CAN MANAGE on the job, the SP has exactly what the pipeline needs (READ VOLUME on landing, WRITE TABLE on target) — least privilege enforced at runtime, fully auditable.

### 7.3 Git integration
- **Configuration:** Job → **Git** dialog: *Git repository URL*, *Git provider* (GitHub, Azure DevOps, GitLab…), *Git reference (branch / tag / commit)* — e.g. `main`, `release-1.1`, or a SHA. Tasks then point at repo-relative paths instead of workspace paths.
- **Runtime behavior:** At each trigger, Jobs **checks out the configured ref at run start** and executes that exact code; the run's history records the commit. A 2 AM run of `main` executes what `main` pointed at when the run began, even if someone pushes 30 seconds later — and any historical run is reproducible by checking out its recorded commit. (A **repair** run uses the *current* code, not the original commit's.)

### 7.4 Declarative Automation Bundles (DABs)
- **Configuration:** A `databricks.yml` describing targets (dev/staging/prod) with `resources.jobs.<name>` — tasks, triggers, environments, notifications, permissions — deployed with `bundle deploy --target <env>`.
- **Runtime behavior:** Deployment is fully repeatable: the same bundle deploys the same job definition into multiple workspaces with per-target overrides (different cron, different run-as SP), and CI (GitHub Actions / GitLab / Jenkins) runs `bundle validate` + `deploy` on merge. Job behavior becomes a pure function of committed YAML — no drift from manual UI edits; rollback is `git revert` + redeploy.

```yaml
# Example bundle job (databricks.yml excerpt)
resources:
  jobs:
    nightly_etl:
      name: nightly_etl
      schedule:
        quartz_cron_expression: "0 0 2 * * ?"
        timezone_id: "America/New_York"
      email_notifications:
        on_failure: [data-oncall@example.com]
      tasks:
        - task_key: ingest
          notebook_task:
            notebook_path: ../src/ingest.ipynb
          environment_key: default
      environments:
        - environment_key: default
          spec:
            environment_version: "2"
            dependencies: ["simplejson==3.19.*"]
```

## 8. Cheat sheet — what happens at runtime

| You configure… | If the condition hits… | Result at runtime |
| --- | --- | --- |
| Cron + max concurrent runs = 1 | Next tick while a run is active | New run **skipped** |
| File arrival trigger | New file lands in the volume | Run starts within seconds |
| Table update trigger | Source table commit | Dependent job triggered |
| Continuous trigger | Run finishes | Next run starts immediately |
| Shared job compute | First task using it starts | Cluster starts; terminates after last task using it |
| Environment spec dependencies | Task starts | Dependencies resolved before code runs |
| If/else condition true | Upstream `row_count > 0` | Only the **(true)** branch runs; (false) skipped |
| Task value set → later task | Task completes | Value readable via `{{tasks.<task>.values.<key>}}` |
| Duration **warning** exceeded | Slow task | Notification only — **keeps running** |
| **Timeout** exceeded | Hung task | Task **killed** → `TIMED_OUT`; downstream skipped |
| Task retries = 3 | Task failure | Up to 3 re-runs; timeout applies per attempt |
| Repair run | Failed multi-task run | Only failed + dependent tasks re-run, on current settings |
| Run-as service principal | Every task | Runs with the SP's UC grants, not the editor's |
| Git reference `main` | Each trigger | Code checked out at run start; commit recorded |

*Sources: Databricks docs — jobs/triggers, jobs/compute, jobs/control-flow, jobs/dynamic-value-references, jobs/configure-task, jobs/repair-job-failures, jobs/notifications, jobs/continuous, admin/system-tables/jobs.*